### DDPM(denoising diffusion probability models)

一个分布可以通过不断地添加噪声变成另一个分布，即来自训练集地图像可以通过不断添加噪声变成符合标准正态分布地图像。
- 正向过程：不断添加高斯噪声，最终变成标准正态分布；
- 反向过程：可学习的神经网络，实现逆操作——去噪（神经网络更擅长去噪任务学习）

### **前向过程**

前向过程是一个马尔可夫过程，这一时刻的图像$x_t$是由上一时刻的图像$x_{t-1}$生成的，即从一个均值与上一时刻图像相关的正态分布中采样得到：
$$x_t \sim N(\mu_t(x_{t-1}), \sigma^2_t I)$$

一般地，设置成：
$$x_t \sim N(\sqrt{1-\beta_t}x_{t-1}, \beta_t I)$$
加噪声能够从慢到快地改变原图像，让图像最终均值为0，方差为I

---
\*因为，此时$x_{t-1}$是已知常量，$\epsilon_{t-1}$是标准正态分布，故$x_t$也是正态分布，均值标准差都可以计算：
$$
\begin{aligned}
x_t &\sim N(\sqrt{1-\beta_t}x_{t-1}, \beta_t I) && \text{(从$x_t$倒推)}\\
\Rightarrow
x_t &= \sqrt{1-\beta_t}x_{t-1} + \sqrt{\beta_t}\epsilon_{t-1} && \text{($\epsilon_{t-1} \sim N(0,1)$)} \\
&= \sqrt{1-\beta_t}(\sqrt{1-\beta_{t-1}}x_{t-2} + \sqrt{\beta_{t-1}}\epsilon_{t-2}) + \sqrt{\beta_t}\epsilon_{t-1} \\
&= \sqrt{(1-\beta_t)(1-\beta_{t-1})}x_{t-2} + \sqrt{(1-\beta_t)\beta_{t-1}}\epsilon_{t-2}+\sqrt{\beta_t}\epsilon_{t-1} \\
&= \sqrt{(1-\beta_t)(1-\beta_{t-1})}x_{t-2} + \sqrt{(1-\beta_t)\beta_{t-1}+\beta_t}\epsilon && \text{(两个独立的正态分布可加)} \\
&= \sqrt{(1-\beta_t)(1-\beta_{t-1})}x_{t-2} + \sqrt{1- (1-\beta_t)(1-\beta_{t-1})}\epsilon && \text{(变形)}\\
&= \sqrt{(1-\beta_t)(1-\beta_{t-1})(1-\beta_{t-2})}x_{t-3} + \sqrt{1- (1-\beta_t)(1-\beta_{t-1})(1-\beta_{t-2})}\epsilon    \\
&=\ldots= \sqrt{\bar \alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\epsilon && \text{(令$\alpha_t=1-\beta_t, \bar\alpha_t = \prod_{i=1}^{t}\alpha_i$)}\\
\end{aligned}
$$
\** 这里的$\beta_t$是一个小于1的常数，（比如从0.0001到0.02线性增长），随着$\beta_t$变大，$\alpha_t$也越小，$\bar\alpha_t$趋于0的速度越快，最后$\bar\alpha_T$几乎为0，此时代入后$X_T$就满足标准正态分布了，符合对于扩散模型的要求（即加噪声直至标准正态）

---
<div style="display: flex; gap: 20px;">
  <div style="flex: 0.5;">

### **Algorithm 1** Training
1. **repeat**:
2. &ensp; $x_0 \sim q(x_0)$
3. &ensp; $t \sim Uniform({1,...,T})$
4. &ensp; $\epsilon \sim N(0,1)$
5. &ensp; Take gradient descent step on $$\nabla_{\theta}\Vert \epsilon-\epsilon_{\theta}(\sqrt{\bar\alpha_t}x_0 + \sqrt{1 - \bar\alpha_t}\epsilon, t) \Vert^2$$
6. **until** converged

  </div>
<div style="flex: 1;">

###
> 2. 从训练集中取出数据$X_0$；
> 3. 随机从(1,...,T)里取一个时刻用于训练，实际训练时不需要一轮预测T个结果，只需要随机预测某一个；
> 4. 随机生成一个噪声$\epsilon$，用于执行前向过程生成$x_T = \sqrt{\bar\alpha_t}x_0 + \sqrt{1 - \bar\alpha_t}\epsilon$；
> 5. 把$x_t$和$t$传给神经网络$\epsilon_{\theta}(x_t, t)$，预测随机噪声。损失函数时预测噪声和实际噪声之间的均方误差，对损失函数采用梯度下降优化网络。

  </div>
</div>

---

### **反向过程**

反向过程中，希望能够倒过来取消每一步加噪声的操作，让一副纯噪声图像变回数据集里的图像。

去噪过程业满足正态分布：
$$x_{t-1} \sim N(\tilde \mu_t, \tilde \beta_t I)$$
\* 因为，当$\beta_t \ll 1$时，$x_t = x_{t-1} + tiny noise$，本质是此时分布方差很小，只有在$x_{t-1} \approx x_t$附近才有概率采样。

因此，神经网络应该输入$t$、$x_t$，拟合当前的**均值$\tilde \mu_t$**和**方差$\tilde \beta_t$**：

在给定了某个训练集输入$x_0$后，由贝叶斯公式：
$$q(x_{t-1} | x_t,x_0) = q(x_t | x_{t-1},x_0)\frac{q(x_{t-1}|x_0)}{q(x_t|x_0)}$$
左式的$q(x_{t-1} | x_t,x_0) = N(x_{t-1}; \tilde\mu_t,\tilde\beta_tI)$表示加噪声的逆操作，其均值和方差都是待求的；<br>
右式的$q(x_t | x_{t-1},x_0) = N(x_t; \sqrt{1-\beta_t}x_{t-1}, \beta_tI)$表示加噪声的分布；<br>
$q(x_{t-1}|x_0)$和$q(x_t|x_0)$两项从$x_0$开始加噪声的连续过程，也是已知的。<br>
即，右式都是已知的，所以可以算出给定$x_0$时的去噪分布。

\* 看成$x_{t-1}$的函数，整个过程仍是马尔可夫，
$$q(x_{t-1} | x_t,x_0) \propto q(x_t|x_{t-1})q(x_{t-1}|x_0)$$
右式本质是两个高斯项相乘，仍为高斯项，其指数项展开后配方即可得到$\tilde \mu_t$和$\tilde \beta_t$:
$$
\begin{aligned}
\tilde\mu_t &= \frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}x_t + \frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1-\bar\alpha_t}x_0 \\
&= \frac{1}{\sqrt{\alpha_t}}(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\epsilon_t) &&\text{(根据前向的闭式表达$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$)}
\end{aligned}
$$

$$\tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t} \cdot \beta_t$$
注意，$\beta_t$是加噪声的方差，是一个常量，那么加噪声的逆操作的方差$\tilde\beta_t$也是一个常量，不与输入$x_0$相关，故训练去噪网络时，只需要拟合 T 个均值。

---

知道均值和方差之后，问题是：如何设置训练的**损失函数**。

根据均值的公式，其中$x_t$是已知的，唯一不确定的只有$\epsilon_t$，故神经网络可以直接预测一个噪声$\epsilon_{\theta}(x_t,t)$（其中$\theta$是可以学习参数），让它和生成$x_t$的噪声$\epsilon_t$的均方误差最小，损失函数为：
$$L = \Vert \epsilon_t - \epsilon_{\theta}(x_t,t) \Vert^2$$
此时，由于每一步的$\epsilon \sim N(0,I)$分布固定，网络始终在拟合同一个目标分布，优化稳定。

---

<div style="display: flex; gap: 20px;">
  <div style="flex: 0.5;">

### **Algorithm 2** Sampling
1. $x_T \sim N(0,1)$
2. **for** t = T,...,1 **do**
3. &ensp; $z \sim N(0,I)$ if $t > 1$, else $z = 0$
4. &ensp; $x_{t-1} = \frac{1}{\sqrt{\alpha_t}}(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\epsilon_{\theta}(x_t,t))+\sigma_t z$
5. **end for**
6. **return** $x_0$


  </div>
  <div style="flex: 1;">

###
> - $x_T$是从标准正态分布中随机采样的输入噪声，要生成不同的图像，只需要更换这个噪声；
> - 接下来，即为**反向过程**：
> - 令时刻从 T 到 1，计算这一时刻去噪声操作的均值和方差，并采样出$x_{t-1}$
> - 均值：$$\mu_{\theta}(x_t,t) = \frac{1}{\sqrt{\alpha_t}}(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\epsilon_{\theta}(x_t,t))$$
> - 方差：$$\sigma^2_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t} \cdot \beta_t$$
> - 最终生成的$x_0$就是生成的图像。

  </div>
</div>

复现基于U-Net的DDPM，并在MNIST数据集上训练：

In [7]:
import torchvision
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, Lambda, ToTensor
import torch

In [4]:
def download_dataset():
    mnist = torchvision.datasets.MNIST(root=r'D:\agent\diffusion model\data\mnist', download=True)
    print('length of MNIST', len(mnist))
    id = 4
    img, label = mnist[id]
    print(img)
    print(label)

    img.show()
    img.save('tmp.jpg')
    tensor = ToTensor()(img)
    print(tensor.shape)
    print(tensor.max())
    print(tensor.min())

if __name__ == '__main__':
    download_dataset()

length of MNIST 60000
<PIL.Image.Image image mode=L size=28x28 at 0x294F3CECCA0>
9
torch.Size([1, 28, 28])
tensor(1.)
tensor(0.)


In [6]:
def get_dataloader(batch_size: int):
    transform = Compose([ToTensor(), Lambda(lambda x: (x - 0.5) * 2)])  # DDPM会把图像和正态分布联系起来，希望取值范围是[-1,1]，故进行线性变换
    dataset = torchvision.datasets.MNIST(root=r'D:\agent\diffusion model\data\mnist',
                                         transform=transform)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
class DDPM():
    def __init__(
            self,
            device: torch.device,
            n_steps: int,   # 时间步T
            min_beta: float = 0.0001,
            max_beta: float = 0.02,
    ):
        """
        初始化，线性生成每个时刻的beta，根据公式计算每个时刻对应的alpha和alpha_bar
        """
        betas = torch.linspace(min_beta, max_beta, n_steps).to(device)
        alphas = 1 - betas
        alpha_bars = torch.empty_like(alphas)
        product = 1
        for i, alpha in enumerate(alphas):
            product *= alpha
            alpha_bars[i] = product
        self.betas = betas
        self.n_steps = n_steps
        self.alphas = alphas
        self.alpha_bars = alpha_bars

